# NordikBank AML: Feature Engineering

# 1. Exploratory Data Analysis (EDA)

Before engineering any features, we explore the raw data to understand its structure,
quality, and distributions. This ensures that all feature engineering decisions are 
informed by the actual data rather than assumptions.
## 1.1 Setup & Data Loading

We begin by importing all required libraries and loading the six case datasets.
All tables are read from the shared `data/` directory. We confirm row and column 
counts for each table to verify successful loading before any analysis begins.

In [3]:
import pandas as pd

# Load all tables
customers = pd.read_csv('../../data/customers.csv')
accounts = pd.read_csv('../../data/accounts.csv')
transactions = pd.read_csv('../../data/transactions.csv', parse_dates=['timestamp'])
baselines = pd.read_csv('../../data/baselines.csv')
alerts = pd.read_csv('../../data/alert_history.csv')
country_risk = pd.read_csv('../../data/country_risk.csv')

# Quick shape check
for name, df in [('customers', customers), ('accounts', accounts), 
                 ('transactions', transactions), ('baselines', baselines),
                 ('alerts', alerts), ('country_risk', country_risk)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols")

customers: 1200 rows, 16 cols
accounts: 2702 rows, 8 cols
transactions: 77384 rows, 15 cols
baselines: 1200 rows, 10 cols
alerts: 19469 rows, 6 cols
country_risk: 48 rows, 4 cols


## 1.2. Target Distribution

We check the class balance of the target variable `suspicious_activity_confirmed` 
across train and val splits. Since suspicious customers are rare by nature, 
understanding the base rate is essential - a high class imbalance risks the model 
defaulting to always predicting "clean", making it useless for detection.

In [5]:
# Filter to labeled rows only (train and val)
labeled = customers[customers['split'].isin(['train', 'val'])]

# Overall target distribution
print("=== Overall target distribution (train + val) ===")
print(labeled['suspicious_activity_confirmed'].value_counts())
print(f"\nBase rate: {labeled['suspicious_activity_confirmed'].mean():.1%}")

# By split
print("\n=== By split ===")
print(labeled.groupby('split')['suspicious_activity_confirmed'].agg(['sum', 'count', 'mean']).rename(columns={'sum': 'suspicious', 'count': 'total', 'mean': 'base_rate'}))

=== Overall target distribution (train + val) ===
suspicious_activity_confirmed
0.0    670
1.0     30
Name: count, dtype: int64

Base rate: 4.3%

=== By split ===
       suspicious  total  base_rate
split                              
train        21.0    500      0.042
val           9.0    200      0.045


## 1.3. Null Profiling

We check null rates across all six tables to identify missing data patterns.
Some nulls are expected by design (e.g. age and income for corporate customers),
while unexpected nulls could silently corrupt features, leading to a model 
trained on incomplete or misleading information.

In [6]:
print("=== Null rates by table ===\n")
for name, df in [('customers', customers), ('accounts', accounts),
                 ('transactions', transactions), ('baselines', baselines),
                 ('alerts', alerts), ('country_risk', country_risk)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f"{name}: no nulls\n")
    else:
        print(f"{name}:")
        for col, n in nulls.items():
            print(f"  {col}: {n} nulls ({n/len(df):.1%})")
        print()

=== Null rates by table ===

customers:
  age: 104 nulls (8.7%)
  occupation_category: 100 nulls (8.3%)
  declared_annual_income: 104 nulls (8.7%)
  declared_annual_turnover: 922 nulls (76.8%)
  industry_code: 922 nulls (76.8%)
  suspicious_activity_confirmed: 500 nulls (41.7%)

accounts: no nulls

transactions:
  counterparty_id: 22808 nulls (29.5%)
  counterparty_bank_country: 74814 nulls (96.7%)
  counterparty_name_hash: 22808 nulls (29.5%)
  merchant_category_code: 62327 nulls (80.5%)

baselines: no nulls

alerts: no nulls

country_risk: no nulls



## 1.4. Customer Type Breakdown

We profile the four customer types (personal, corporate, sole_trader, SME) and 
their suspicious activity rates. Different customer types have fundamentally 
different transaction behaviors and risk profiles, so understanding their 
distribution ensures we never mistakenly treat them as the same during feature engineering.

In [7]:
print("=== Customer type distribution ===")
print(customers['customer_type'].value_counts())

print("\n=== Suspicious rate by customer type (train + val only) ===")
labeled = customers[customers['split'].isin(['train', 'val'])]
print(labeled.groupby('customer_type')['suspicious_activity_confirmed']
      .agg(['sum', 'count', 'mean'])
      .rename(columns={'sum': 'suspicious', 'count': 'total', 'mean': 'base_rate'})
      .round(3))

=== Customer type distribution ===
customer_type
personal       864
sole_trader    147
corporate      100
SME             89
Name: count, dtype: int64

=== Suspicious rate by customer type (train + val only) ===
               suspicious  total  base_rate
customer_type                              
SME                   2.0     48      0.042
corporate             8.0     61      0.131
personal             13.0    499      0.026
sole_trader           7.0     92      0.076


## 1.5 Transaction Profiling

We examine the distribution of transaction types, channels, statuses, and currencies.
This reveals which transaction categories dominate the dataset and flags any 
unusual patterns, such as high declined rates, that are relevant AML signals 
and will inform which features we engineer.

In [8]:
print("=== Transaction types ===")
print(transactions['transaction_type'].value_counts())

print("\n=== Channels ===")
print(transactions['channel'].value_counts())

print("\n=== Status ===")
print(transactions['status'].value_counts())

print("\n=== Currencies ===")
print(transactions['currency'].value_counts())

print("\n=== Amount range (approved only) ===")
approved = transactions[transactions['status'] == 'approved']
print(approved['amount'].describe().round(2))

=== Transaction types ===
transaction_type
transfer              22995
direct_debit          20840
card_payment          15057
standing_order         8264
cash_deposit           4603
cash_withdrawal        3195
international_wire     2430
Name: count, dtype: int64

=== Channels ===
channel
online_banking    57197
mobile_app        12386
branch             4425
ATM                3376
Name: count, dtype: int64

=== Status ===
status
approved    75807
declined     1577
Name: count, dtype: int64

=== Currencies ===
currency
DKK    76357
USD      536
EUR      491
Name: count, dtype: int64

=== Amount range (approved only) ===
count      75807.00
mean       45412.62
std       307451.32
min     -2925988.89
25%         -678.88
50%         -247.46
75%         7600.00
max      7328554.68
Name: amount, dtype: float64


## 1.6 Join Sanity Checks

We verify that all foreign keys across tables are consistent, ensuring every 
transaction and account links back to a valid customer. Broken joins would mean 
we silently lose data during feature engineering, leading to incomplete customer profiles.

In [9]:
# Check all transaction customer_ids exist in customers
txn_customers = set(transactions['customer_id'].unique())
all_customers = set(customers['customer_id'].unique())

missing_in_customers = txn_customers - all_customers
print(f"Transaction customer_ids not in customers: {len(missing_in_customers)}")

# Check all account customer_ids exist in customers
acc_customers = set(accounts['customer_id'].unique())
missing_acc = acc_customers - all_customers
print(f"Account customer_ids not in customers: {len(missing_acc)}")

# Check all alert customer_ids exist in customers
alert_customers = set(alerts['customer_id'].unique())
missing_alerts = alert_customers - all_customers
print(f"Alert customer_ids not in customers: {len(missing_alerts)}")

# Check all baseline customer_ids exist in customers
base_customers = set(baselines['customer_id'].unique())
missing_base = base_customers - all_customers
print(f"Baseline customer_ids not in customers: {len(missing_base)}")

# Check customers with no transactions
no_txn = all_customers - txn_customers
print(f"\nCustomers with no transactions: {len(no_txn)}")

Transaction customer_ids not in customers: 0
Account customer_ids not in customers: 0
Alert customer_ids not in customers: 0
Baseline customer_ids not in customers: 0

Customers with no transactions: 0


## 1.7 Time Coverage

We verify that transactions span the expected Jan–Dec 2024 window and check 
whether coverage is uniform across customers. Gaps in transaction history for 
specific customers could indicate dormancy, itself an AML signal, or data 
quality issues that affect feature reliability.

In [10]:
print("=== Overall date range ===")
print(f"Min: {transactions['timestamp'].min()}")
print(f"Max: {transactions['timestamp'].max()}")

print("\n=== Transactions per month ===")
transactions['month'] = transactions['timestamp'].dt.to_period('M')
print(transactions.groupby('month').size())

print("\n=== Transactions per customer (approved only) ===")
approved = transactions[transactions['status'] == 'approved']
txn_per_customer = approved.groupby('customer_id').size()
print(txn_per_customer.describe().round(2))

print(f"\nCustomers with fewer than 5 transactions: {(txn_per_customer < 5).sum()}")

=== Overall date range ===
Min: 2024-01-01 00:00:00
Max: 2024-12-31 22:57:00

=== Transactions per month ===
month
2024-01    6170
2024-02    6424
2024-03    6245
2024-04    6357
2024-05    6424
2024-06    6334
2024-07    6317
2024-08    6319
2024-09    6486
2024-10    6607
2024-11    6668
2024-12    7033
Freq: M, dtype: int64

=== Transactions per customer (approved only) ===
count    1200.00
mean       63.17
std        28.10
min        18.00
25%        47.00
50%        58.00
75%        70.00
max       206.00
dtype: float64

Customers with fewer than 5 transactions: 0


## 1.8 Country Risk Coverage

We check how many unique counterparty countries appear in the transaction data 
and what share successfully join to the country_risk table. This is critical 
because geographic risk is a core AML signal. Transactions routed through 
high-risk or FATF-listed countries must be captured reliably in our features.

In [11]:
# Unique counterparty countries in transactions
unique_countries = transactions['counterparty_bank_country'].dropna().unique()
print(f"Unique counterparty countries in transactions: {len(unique_countries)}")

# How many join to country_risk
risk_countries = set(country_risk['country_code'].unique())
matched = [c for c in unique_countries if c in risk_countries]
unmatched = [c for c in unique_countries if c not in risk_countries]

print(f"Matched to country_risk: {len(matched)}")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print(f"Unmatched countries: {unmatched}")

print(f"\n=== FATF status distribution in country_risk ===")
print(country_risk['fatf_status'].value_counts())

print(f"\n=== EU high risk list ===")
print(country_risk['eu_high_risk_list'].value_counts())

Unique counterparty countries in transactions: 36
Matched to country_risk: 36
Unmatched: 0

=== FATF status distribution in country_risk ===
fatf_status
compliant              32
enhanced_monitoring    11
high_risk               4
blacklisted             1
Name: count, dtype: int64

=== EU high risk list ===
eu_high_risk_list
False    40
True      8
Name: count, dtype: int64


# 2. Feature Engineering

**Goal:** Transform raw transaction data into a single customer-level feature table 
(one row per 1,200 customers) that captures behavioral signals indicative of 
money laundering, making it ready for model training.

We transform the raw transaction data into meaningful customer-level behavioral 
features. Each feature is designed to capture a specific AML signal that a 
compliance analyst would recognise as indicative of suspicious activity.
The output is a single feature table at customer grain, one row per customer,
ready for model training.

In [12]:
# Work on approved transactions only for behavioral features
approved = transactions[transactions['status'] == 'approved'].copy()  # exclude declined transactions

# Flag cash transaction types
cash_types = ['cash_deposit', 'cash_withdrawal']  # define what counts as cash
approved['is_cash'] = approved['transaction_type'].isin(cash_types)  # True/False column per transaction

# Customer-level cash intensity features
cash_features = approved.groupby('customer_id').agg(
    total_transactions=('transaction_id', 'count'),  # total approved transactions per customer
    cash_transaction_count=('is_cash', 'sum'),  # number of cash transactions per customer
    cash_volume=('amount', lambda x: x[approved.loc[x.index, 'is_cash']].abs().sum()),  # total cash amount (absolute value)
    total_volume=('amount', lambda x: x.abs().sum()),  # total transaction volume (absolute value)
).reset_index()  # bring customer_id back as a column

# Cash ratio features
cash_features['cash_transaction_ratio'] = (
    cash_features['cash_transaction_count'] / cash_features['total_transactions']  # share of transactions that are cash
)
cash_features['cash_volume_ratio'] = (
    cash_features['cash_volume'] / cash_features['total_volume']  # share of total volume that is cash
)

# Structuring — transactions just below 15,000 DKK threshold
threshold = 15000  # NordikBank's internal reporting threshold
approved['is_structuring_candidate'] = (
    approved['is_cash'] &  # must be a cash transaction
    (approved['amount'].abs() >= threshold * 0.8) &  # at least 80% of threshold (12,000 DKK)
    (approved['amount'].abs() < threshold)  # strictly below threshold
)

# Count structuring candidates per customer
structuring = approved.groupby('customer_id')['is_structuring_candidate'].sum().reset_index()
structuring.columns = ['customer_id', 'structuring_count']  # rename for clarity

# Merge structuring count into cash features
cash_features = cash_features.merge(structuring, on='customer_id', how='left')  # left join to keep all customers

print(cash_features.describe().round(2))

       total_transactions  cash_transaction_count  cash_volume  total_volume  \
count             1200.00                 1200.00      1200.00       1200.00   
mean                63.17                    6.42     28153.52    3271648.50   
std                 28.10                   19.08    113688.60    9148163.69   
min                 18.00                    0.00         0.00      93641.16   
25%                 47.00                    1.00       200.00     384318.76   
50%                 58.00                    2.50      3500.00     588901.86   
75%                 70.00                    5.00      8000.00    1060290.50   
max                206.00                  138.00    794600.00   67916658.92   

       cash_transaction_ratio  cash_volume_ratio  structuring_count  
count                 1200.00            1200.00            1200.00  
mean                     0.08               0.03               0.46  
std                      0.12               0.11               2.01  